In [3]:
from pyspark.sql import functions as F

# 1. Map your two active Bronze tables inside OneLake
INFO_TABLE_PATH = "studentinfo_bronze"
VLE_TABLE_PATH  = "studentvle_bronze"

print("🛰️ Beginning Parallel Fabric Lakehouse Audit for OULAD Tiers...")

try:
    # ----------------------------------------------------
    # AUDIT PART A: STUDENT INFO (DIMENSION RUNTIME)
    # ----------------------------------------------------
    df_info = spark.read.table(INFO_TABLE_PATH)
    total_info_rows = df_info.count()
    print(f"\n📊 [INFO AUDIT] Total Registered Records: {total_info_rows}")
    
    # Structural Primary Key Validation (id_student + code_module + code_presentation)
    info_duplicates = df_info.groupBy("id_student", "code_module", "code_presentation") \
                             .count() \
                             .filter("count > 1") \
                             .count()
    print(f"🚨 [INFO AUDIT] Duplicate Registration Profiles Found: {info_duplicates}")
    
    # ----------------------------------------------------
    # AUDIT PART B: STUDENT VLE (CLICKSTREAM LOG RUNTIME)
    # ----------------------------------------------------
    df_vle = spark.read.table(VLE_TABLE_PATH)
    total_vle_rows = df_vle.count()
    print(f"\n📊 [VLE AUDIT] Total Clickstream Records Loaded: {total_vle_rows}")
    
    # Quality Check: Check for orphaned clicks with missing structural markers
    null_vle_clicks = df_vle.filter(
        F.col("id_student").isNull() | 
        F.col("id_site").isNull() | 
        F.col("sum_click").isNull()
    ).count()
    print(f"🚨 [VLE AUDIT] Corrupted/Null Click Event Packets: {null_vle_clicks}")
    
    # ----------------------------------------------------
    # AUDIT PART C: GLOBAL REPOSITORY METADATA LINEAGE CHECK
    # ----------------------------------------------------
    print("\n📋 Running HCAI Provenance Pipeline Verification...")
    info_missing_meta = df_info.filter(F.col("PIPELINE_INGEST_TS").isNull() | F.col("SOURCE_FILE_NAME").isNull()).count()
    vle_missing_meta  = df_vle.filter(F.col("PIPELINE_INGEST_TS").isNull() | F.col("SOURCE_FILE_NAME").isNull()).count()
    
    print(f"✔ Info Table Metadata Missing: {info_missing_meta}")
    print(f"✔ VLE Table Metadata Missing: {vle_missing_meta}")
    
    # ----------------------------------------------------
    # FINAL VERDICT CONFORMANCE GATE
    # ----------------------------------------------------
    print("\n🏁 ===============================================")
    if info_duplicates == 0 and null_vle_clicks == 0 and info_missing_meta == 0 and vle_missing_meta == 0:
        print("🚀 SUCCESS: OULAD Complete Bronze Track Validation Verified Pass!")
    else:
        print("⚠️ WARNING: OULAD Data Quality Framework Alert! Check anomalies reported above.")
    print("==================================================")

except Exception as e:
    print(f"❌ FABRIC WORKSPACE VALIDATION RUNTIME CRITICAL FAILURE: {str(e)}")


StatementMeta(, 876e3113-da94-47c8-ae56-db5518c541a3, 5, Finished, Available, Finished, False)

🛰️ Beginning Parallel Fabric Lakehouse Audit for OULAD Tiers...

📊 [INFO AUDIT] Total Registered Records: 32593
🚨 [INFO AUDIT] Duplicate Registration Profiles Found: 0

📊 [VLE AUDIT] Total Clickstream Records Loaded: 10655280
🚨 [VLE AUDIT] Corrupted/Null Click Event Packets: 0

📋 Running HCAI Provenance Pipeline Verification...
✔ Info Table Metadata Missing: 0
✔ VLE Table Metadata Missing: 0

🏁 ===============================================
🚀 SUCCESS: OULAD Complete Bronze Track Validation Verified Pass!


In [1]:
spark.sql("SHOW TABLES").show()

StatementMeta(, 519d5573-1d25-43ec-9bfa-9a772a949e9b, 3, Finished, Available, Finished, False)

+--------------------+------------------+-----------+
|           namespace|         tableName|isTemporary|
+--------------------+------------------+-----------+
|OULAD_DENG_HCAI_R...|studentinfo_bronze|      false|
|OULAD_DENG_HCAI_R...| studentvle_bronze|      false|
+--------------------+------------------+-----------+

